In [1]:
import torchmetrics

from get_device import get_device
from model_runner.LRReducingModelRunner import LRReducingModelRunner
from models.rnn_models import SimpleRnnModel
from torch.utils.data import DataLoader
from data_loaders.ForecaseAheadDataset import ForecastAheadDataset
from RNN_Sequences.chicago_transit_data import download_and_extract_ridership_data, load_ridership_data
import pandas as pd
import torch

download_and_extract_ridership_data()
df = load_ridership_data()

df_multivariate = df[["rail", "bus"]] / 1e6  # use both rail & bus series as input
df_multivariate["next_day_type"] = df["day_type"].shift(-1)  # we know tomorrow's type
df_multivariate = pd.get_dummies(df_multivariate, dtype=float)  # one-hot encode day typ

multivariate_train = torch.FloatTensor(df_multivariate["2016-01":"2018-12"].values)
multivariate_valid = torch.FloatTensor(df_multivariate["2019-01":"2019-05"].values)
multivariate_test = torch.FloatTensor(df_multivariate["2019-06":].values)

window_length = 56
forecast_length = 14
ahead_train_set = ForecastAheadDataset(multivariate_train, window_length, forecast_length)
ahead_train_loader = DataLoader(ahead_train_set, batch_size=32, shuffle=True)
ahead_valid_set = ForecastAheadDataset(multivariate_valid, window_length, forecast_length)
ahead_valid_loader = DataLoader(ahead_valid_set, batch_size=32)
ahead_test_set = ForecastAheadDataset(multivariate_test, window_length, forecast_length)
ahead_test_loader = DataLoader(ahead_test_set, batch_size=32)

In [2]:
torch.manual_seed(42)
ahead_model = SimpleRnnModel(input_size=5, hidden_size=32, output_size=14).to(get_device())

accuracy = torchmetrics.MeanAbsoluteError()
optimizer = torch.optim.SGD(ahead_model.parameters(), lr=0.05, momentum=0.95)
loss_fn = torch.nn.HuberLoss()
runner = LRReducingModelRunner(ahead_model, accuracy, optimizer=optimizer, loss_fn=loss_fn)

runner.train_model(ahead_train_loader, ahead_valid_loader, n_epochs=75, patience=75)

test_result = runner.test_model(data_loader=ahead_test_loader) * 1e6
print(test_result)


Epoch:1 / 75, train loss: 0.0725, train metric: 0.3051, valid metric: 0.1730 (best),  in 3.5s
Epoch:2 / 75, train loss: 0.0164, train metric: 0.1480, valid metric: 0.1142 (best),  in 0.6s
Epoch:3 / 75, train loss: 0.0099, train metric: 0.1133, valid metric: 0.0975 (best),  in 0.7s
Epoch:4 / 75, train loss: 0.0076, train metric: 0.0946, valid metric: 0.0832 (best),  in 0.7s
Epoch:5 / 75, train loss: 0.0066, train metric: 0.0846, valid metric: 0.0749 (best),  in 0.7s
Epoch:6 / 75, train loss: 0.0060, train metric: 0.0783, valid metric: 0.0651 (best),  in 0.8s


KeyboardInterrupt: 

In [ ]:
test_input, test_target = ahead_test_set[0]
print(test_input.shape, test_target.shape)
print(test_input)

test_output = runner.run_model(test_input)
print(test_output)